In [1]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from pathlib import Path
from sklearn.linear_model import Ridge, ElasticNet
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split,cross_validate,KFold,GridSearchCV,RandomizedSearchCV
from sklearn.metrics import root_mean_squared_log_error
from sklearn.pipeline import Pipeline
from typing import cast
from add_features import add_modified_features

%matplotlib inline

warnings.filterwarnings('ignore')

## Data Preprocessing

### Cast

In [2]:
def mycast(df):
    df['MSSubClass'] = df['MSSubClass'].apply(str)
    df['YrSold'] = df['YrSold'].astype(str)
    df['MoSold'] = df['MoSold'].astype(str)
    return df

### Clean

In [3]:
def clean(df):
    # maybe typo
    df["Exterior2nd"] = df["Exterior2nd"].replace({"Brk Cmn": "BrkComm"})
    df["GarageYrBlt"] = df["GarageYrBlt"].where(df.GarageYrBlt <= 2010, df.YearBuilt)
    return df

### Impute

In [4]:
def impute(df):
    # suitable
    df['Functional'] = df['Functional'].fillna('Typ') 
    df['Electrical'] = df['Electrical'].fillna("SBrkr") 
    df['KitchenQual'] = df['KitchenQual'].fillna("TA") 
    df["PoolQC"] = df["PoolQC"].fillna("None")
    # mode
    df['Exterior1st'] = df['Exterior1st'].fillna(df['Exterior1st'].mode()[0]) 
    df['Exterior2nd'] = df['Exterior2nd'].fillna(df['Exterior2nd'].mode()[0])
    df['SaleType'] = df['SaleType'].fillna(df['SaleType'].mode()[0])
    # zero
    for col in ('GarageYrBlt', 'GarageArea', 'GarageCars'):
        df[col] = df[col].fillna(0)
    # None
    for col in ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']:
        df[col] = df[col].fillna('None')
    for col in ('BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'):
        df[col] = df[col].fillna('None')
    
    # group mean
    # 同じ住宅タイプは同じ地域に集中する(都市計画的な意味で)だろうというアイデアに基づく
    df['MSZoning'] = df.groupby('MSSubClass')['MSZoning'].transform(lambda x: x.fillna(x.mode()[0]))
    
    # 残りの str 型は一律 None
    non_numeric_cols = df.select_dtypes(exclude='number').columns
    df[non_numeric_cols] = df[non_numeric_cols].fillna('None')

    # median
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # 残りの numerics 型は一律 0
    numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    numerics = []
    for i in df.columns:
        if df[i].dtype in numeric_dtypes:
            numerics.append(i)
    df.update(df[numerics].fillna(0))
    return df
        

### load

In [5]:
def load_data(return_concat = False):
    # Read
    dir = Path("../../data/")
    train = pd.read_csv(dir / "train.csv", index_col='Id')
    test = pd.read_csv(dir / "test.csv", index_col='Id')
    # filter outlier
    outlier_ids = [524,1299]  # 確認した外れ値のId
    train = train.drop(outlier_ids)
    # Preprocessing
    df= pd.concat([train, test],)
    df = clean(df)
    df = impute(df)
    df = mycast(df)
    # Reform splits
    train = df.loc[train.index, :]
    test = df.loc[test.index, :]
    # Return concat or split
    if return_concat:
        return df
    else:
        return train, test
    
train, test = cast(tuple[pd.DataFrame,pd.DataFrame],load_data(return_concat=False))
test = test.drop(['SalePrice'],axis=1)
train.shape, test.shape

((1458, 80), (1459, 79))

### prepare

In [6]:
X = train.copy()
y = X.pop('SalePrice')
X_submit = test.copy()

### build pipeline

In [7]:
from preprocess import build_target_transformer, build_preprocessor

log_standardize_y = build_target_transformer()
tree_preprocessor, reg_preprocessor = build_preprocessor()

### add features / train test split

In [8]:
# add modified
X = add_modified_features(pl.DataFrame(X))
X_submit = add_modified_features(pl.DataFrame(X_submit))

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("-"*20 + "train valid shape" + "-"*20)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_va: {X_test.shape}, y_va: {y_test.shape}")

print("-"*20 + "Null Check" + "-"*20)
s = X_train.isnull().sum()
print(s[s > 0])

--------------------train valid shape--------------------
X_train: (1166, 105), y_train: (1166,)
X_va: (292, 105), y_va: (292,)
--------------------Null Check--------------------
Series([], dtype: int64)


## Modeling

### CV Function

In [9]:
folds = 5
cv = KFold(n_splits=folds, shuffle=True, random_state=42)

def show_rmsle(name, model):

    result = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring="neg_root_mean_squared_log_error",
        return_train_score=True,
    )

    # show result
    train_scores = -result["train_score"]
    valid_scores = -result["test_score"]
    model.fit(X_train, y_train)
    test_score = root_mean_squared_log_error(y_test, model.predict(X_test))
    print(f"-"*20 + f"{name} スコア" + "-"*20)
    print(f"CV Ave: train {train_scores.mean():.4f}({train_scores.std():.4f}) / valid {valid_scores.mean():.4f}({valid_scores.std():.4f})")
    print(f"Test: {test_score:.4f}")

### Ridge

In [24]:
alpha = 0.001
base_model = Pipeline([
    ("prep", reg_preprocessor),
    ("ridge", Ridge(alpha=alpha ,random_state=42))
])
ridge = TransformedTargetRegressor(
    regressor = base_model,
    transformer = log_standardize_y,
    check_inverse = False
)

show_rmsle("Ridge", ridge)

--------------------Ridge スコア--------------------
CV Ave: train 0.1065(0.0020) / valid 0.1185(0.0093)
Test: 0.1193


### ENet

In [10]:
alpha = 0.001
l1_ratio = 0.9
base_model = Pipeline([
    ("prep", reg_preprocessor),
    ("enet", ElasticNet(alpha=alpha,l1_ratio=l1_ratio, random_state=42))
])
enet = TransformedTargetRegressor(
    regressor = base_model,
    transformer = log_standardize_y,
    check_inverse = False
)

show_rmsle("ENet", enet)

--------------------ENet スコア--------------------
CV Ave: train 0.1069(0.0020) / valid 0.1174(0.0096)
Test: 0.1187


#### grid search(Enet)

In [11]:
# param_grid = {
#     "regressor__enet__alpha": [0.001, 0.01, 0.1,],
#     "regressor__enet__l1_ratio": [0.5, 0.7, 0.9,]
# }

# grid_search = GridSearchCV(
#     estimator = enet,
#     param_grid=param_grid,
#     scoring="neg_root_mean_squared_log_error",
#     cv=cv,
#     n_jobs=-1,
#     refit=True,
# )
# grid_search.fit(X_train, y_train)

# print(f"Best params: {grid_search.best_params_}")
# print(f"Best CV RMSLE: {-grid_search.best_score_:.4f}")

# best_model = grid_search.best_estimator_
# test_score_gs = root_mean_squared_log_error(y_test, best_model.predict(X_test))
# print(f"最終スコア(test RMSLE): {test_score_gs:.4f}")

# # alphaごとのCVスコア一覧
# cv_results = pd.DataFrame(grid_search.cv_results_)[
#     ["param_regressor__enet__alpha", "param_regressor__enet__l1_ratio", "mean_test_score", "std_test_score"]
# ].assign(
#     mean_test_RMSLE=lambda df: -df["mean_test_score"]
# ).sort_values("param_regressor__enet__alpha")
# cv_results


### XGBoost

In [12]:
params = dict(
    max_depth=6,           # maximum depth of each tree - try 2 to 10
    learning_rate=0.01,    # effect of each tree - try 0.0001 to 0.1
    n_estimators=1000,     # number of trees (that is, boosting rounds) - try 1000 to 8000
    min_child_weight=5,    # minimum number of houses in a leaf - try 1 to 10
    colsample_bytree=0.5,  # fraction of features (columns) per tree - try 0.2 to 1.0
    subsample=0.7,         # fraction of instances (rows) per tree - try 0.2 to 1.0
    reg_alpha=0.01,         # L1 regularization (like LASSO) - try 0.0 to 10.0
    reg_lambda=10,        # L2 regularization (like Ridge) - try 0.0 to 10.0
    num_parallel_tree=1,   # set > 1 for boosted random forests
    random_state=42,
    n_jobs=-1
)
base_model = Pipeline([
    ("prep", tree_preprocessor),
    ("xgb", XGBRegressor(**params))
])
xgb = TransformedTargetRegressor(
    regressor = base_model,
    transformer = log_standardize_y,
    check_inverse = False
)

show_rmsle("XGB", xgb)

--------------------XGB スコア--------------------
CV Ave: train 0.1166(0.0014) / valid 0.1735(0.0091)
Test: 0.1726


In [13]:
# param_dist_xgb = {
#     "regressor__xgb__n_estimators": [800, 1000],
#     "regressor__xgb__learning_rate": [0.01, 0.02, 0.05],
#     "regressor__xgb__max_leaves": [5, 7],
#     "regressor__xgb__max_depth": [3, 4, 5],
#     "regressor__xgb__min_child_weight": [1, 5, 10],
#     "regressor__xgb__reg_alpha": [0.01, 0.05, 0.1],
#     "regressor__xgb__reg_lambda": [1.0, 5.0, 10.0],
#     "regressor__xgb__subsample": [0.2, 0.5, 0.7],
#     "regressor__xgb__colsample_bytree": [0.5, 0.6, 0.7],
# }

# random_search_xgb = RandomizedSearchCV(
#     estimator=xgb,
#     param_distributions=param_dist_xgb,
#     scoring="neg_root_mean_squared_log_error",
#     cv=cv,
#     n_iter=40,
#     n_jobs=-1,
#     refit=True,
#     random_state=42,
# )
# random_search_xgb.fit(X_train, y_train)

# print(f"Best params: {random_search_xgb.best_params_}")
# print(f"Best CV RMSLE: {-random_search_xgb.best_score_:.4f}")


# # パラメータごとのCVスコア一覧
# cv_results_xgb = pd.DataFrame(random_search_xgb.cv_results_)[
#     [
#         "param_regressor__xgb__n_estimators",
#         "param_regressor__xgb__learning_rate",
#         "param_regressor__xgb__max_leaves",
#         "param_regressor__xgb__max_depth",
#         "param_regressor__xgb__min_child_weight",
#         "param_regressor__xgb__reg_alpha",
#         "param_regressor__xgb__reg_lambda",
#         "mean_test_score",
#         "std_test_score"
#     ]
# ].assign(
#     mean_test_RMSLE=lambda df: -df["mean_test_score"]

# ).sort_values("mean_test_RMSLE")
# cv_results_xgb.sort_values('mean_test_RMSLE',ascending=True).head(15)

In [14]:
# # 特徴量の寄与度（XGBoost feature_importances_）トップ25
# all_feature_names_xgb = xgb.regressor_.named_steps["prep"].get_feature_names_out()
# # selected_mask_xgb = best_model_xgb.regressor_.named_steps["select"].get_support()
# feature_names_xgb = all_feature_names_xgb  # [selected_mask_xgb]
# importances_xgb = xgb.regressor_.named_steps["xgb"].feature_importances_

# importance_df_xgb = pl.DataFrame({
#     "feature": feature_names_xgb,
#     "importance": importances_xgb,
# }).sort("importance", descending=True)

# importance_df_xgb = importance_df_xgb.head(25)

# plt.figure(figsize=(9, 10))
# sns.barplot(
#     data=importance_df_xgb.to_pandas(),
#     x="importance", y="feature",
#     hue="importance", palette="coolwarm", dodge=False, legend=False,
# )
# plt.title("Top 25 of XGBoost feature_importances_")
# plt.xlabel("importance")
# plt.ylabel("df")
# plt.tight_layout()

# plt.show()


### SVR (kernel=rbf)

In [15]:
base_model_svr = SVR(
        kernel="rbf",
        C=10,
        epsilon=0.05,
        gamma=0.0005,
    )
base_model = Pipeline([
    ("prep", reg_preprocessor),
    ("svr", base_model_svr)
])
svr = TransformedTargetRegressor(
    regressor = base_model,
    transformer = log_standardize_y,
    check_inverse = False
)

show_rmsle("svr", svr)

--------------------svr スコア--------------------
CV Ave: train 0.1034(0.0020) / valid 0.1160(0.0112)
Test: 0.1184


#### grid search (SVR)

In [16]:
# param_dist_svr = {
#     "regressor__svr__C": [0.5, 1.0, 3.0, 5.0, 10.0],
#     "regressor__svr__epsilon": [0.01, 0.02, 0.05],
#     "regressor__svr__gamma": [0.0005, 0.001, 0.005, 0.01],
# }

# random_search_svr = RandomizedSearchCV(
#     estimator=svr,
#     param_distributions=param_dist_svr,
#     scoring="neg_root_mean_squared_log_error",
#     cv=cv,
#     n_iter=100,
#     n_jobs=-1,
#     refit=True,
#     random_state=42,
# )
# random_search_svr.fit(X_train, y_train)

# print(f"Best params: {random_search_svr.best_params_}")
# print(f"Best CV RMSLE: {-random_search_svr.best_score_:.4f}")

# best_model_svr = random_search_svr.best_estimator_
# test_score_gs_svr = root_mean_squared_log_error(y_test, best_model_svr.predict(X_test))
# print(f"最終スコア(test RMSLE): {test_score_gs_svr:.4f}")

# # パラメータごとのCVスコア一覧
# cv_results_svr = pd.DataFrame(random_search_svr.cv_results_)[
#     [
#         "param_regressor__svr__C",
#         "param_regressor__svr__epsilon",
#         "param_regressor__svr__gamma",
#         "mean_test_score",
#         "std_test_score"
#     ]
# ].assign(
#     mean_test_RMSLE=lambda df: -df["mean_test_score"]

# ).sort_values("mean_test_RMSLE")
# cv_results_svr.sort_values('mean_test_RMSLE',ascending=True).head(15)

### stacking

In [17]:
from sklearn.ensemble import StackingRegressor
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone

class AveragingModels(BaseEstimator, RegressorMixin, TransformerMixin):
    """ 複数モデルの予測値の単純平均を予測とするクラス """
    
    def __init__(self, models):
        self.models = models

    def fit(self, X, y):
        self.models_ = [clone(x) for x in self.models]
        
        for model in self.models_:
            model.fit(X, y)

        return self
    
    def predict(self, X):
        predictions = np.column_stack([
            model.predict(X) for model in self.models_
        ])
        return np.mean(predictions, axis=1)   

In [18]:
averaged_models = AveragingModels(models = (enet, xgb, svr))
show_rmsle("simple average",averaged_models)

--------------------simple average スコア--------------------
CV Ave: train 0.0981(0.0019) / valid 0.1209(0.0093)
Test: 0.1251


### submit

In [25]:
pred_final = ridge.predict(X_submit)

pd.DataFrame({
    "Id": test.index,
    "SalePrice": pred_final
}).to_csv("../../data/temp_ridge_submission(base_v2).csv",index=False)